In [ ]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']: os.environ[v]='1'
import pandas as pd, numpy as np, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
# Heavy models (gradient boosting, XGBoost, EBM) are OFF by default because they can crash the MyDRE kernel
# on large memory use. Turn this on only once the environment is confirmed stable.
RUN_HEAVY = False
HAVE_HGB=HAVE_XGB=HAVE_EBM=False
if RUN_HEAVY:
    try:
        from sklearn.ensemble import HistGradientBoostingClassifier; HAVE_HGB=True
    except Exception: HAVE_HGB=False
    try:
        from xgboost import XGBClassifier; HAVE_XGB=True
    except Exception: HAVE_XGB=False
    try:
        from interpret.glassbox import ExplainableBoostingClassifier; HAVE_EBM=True
    except Exception: HAVE_EBM=False
np.random.seed(42)
THESIS=Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis")
DERIVED=THESIS/"data_derived"; OUT=THESIS/"outputs"/"modelling"; OUT.mkdir(parents=True, exist_ok=True)
df=pd.read_csv(DERIVED/"cohort_clean.csv")
df['textbook_failure']=1-pd.to_numeric(df['textbook_outcome'],errors='coerce')   # adverse direction for an interpretable decision curve
print('cohort',df.shape,'| RUN_HEAVY',RUN_HEAVY,'| XGBoost',HAVE_XGB,'| EBM',HAVE_EBM)

In [ ]:
PREOP=['age_at_surgery','sex_male','bmi','asascore','comlong','charlson_cci','anamok_prior_surgery',
       'immunosuppression','neoadj_received','neoadj_chemoradiation','ct_stage','cn_stage','histology_adeno']
OPER_CORE=['surgical_approach_mis','anastomosis_cervical','resection_transthoracic','conversion']
OPER_EXT=['op_duration_min','blood_loss_ml']
CONT=['age_at_surgery','bmi','charlson_cci','ct_stage','cn_stage','op_duration_min','blood_loss_ml']
OUTCOMES=['pulmonary','cd_ge_IIIb','reintervention','los_prolonged','textbook_failure']
HEADLINE=['cd_ge_IIIb','los_prolonged']
PREOP=[c for c in PREOP if c in df.columns]; OPER_CORE=[c for c in OPER_CORE if c in df.columns]
print('preop',len(PREOP),'oper_core',len(OPER_CORE))

In [ ]:
def pipe_fixed(cols,penalty='l2',C=1.0):
    cont=[c for c in CONT if c in cols]; pre=ColumnTransformer([('s',StandardScaler(),cont)],remainder='passthrough')
    return Pipeline([('t',pre),('lr',LogisticRegression(penalty=penalty,C=C,solver='liblinear',max_iter=5000,random_state=42))])
def best_C(X,y):
    # tune the L2 penalty once on log loss; light, avoids nested cross validation that strains the kernel
    cont=[c for c in CONT if c in X.columns]; pre=ColumnTransformer([('s',StandardScaler(),cont)],remainder='passthrough')
    m=LogisticRegressionCV(Cs=10,cv=5,scoring='neg_log_loss',penalty='l2',solver='liblinear',max_iter=5000,random_state=42)
    Pipeline([('t',pre),('lr',m)]).fit(X,y); return float(m.C_[0])
def pipe_tuned(cols,C=1.0): return pipe_fixed(cols,'l2',C)
def cal_slope(y,p):
    eps=1e-6; p=np.clip(p,eps,1-eps); lg=np.log(p/(1-p))
    return float(LogisticRegression(penalty='l2',C=1e8,solver='lbfgs',max_iter=3000).fit(lg.reshape(-1,1),y).coef_[0,0])
def evalm(X,y,tuned=True,nb=200,rs=42):
    C=best_C(X,y) if tuned else 1.0
    pipe=pipe_fixed(list(X.columns),'l2',C)
    yp=cross_val_predict(pipe,X,y,cv=StratifiedKFold(5,shuffle=True,random_state=rs),method='predict_proba')[:,1]; yv=np.asarray(y)
    rng=np.random.RandomState(rs); b=[roc_auc_score(yv[i],yp[i]) for i in (rng.choice(len(yv),len(yv),True) for _ in range(nb)) if len(np.unique(yv[i]))>1]
    b=np.array(b)
    return {'auc':roc_auc_score(yv,yp),'lo':np.percentile(b,2.5),'hi':np.percentile(b,97.5),'boot':b,'cvp':yp,'y':yv,
            'ap':average_precision_score(yv,yp),'brier':brier_score_loss(yv,yp),'cal':cal_slope(yv,yp),'mp':float(yp.mean()),'obs':float(yv.mean())}
def dca(y,p,ts):
    y=np.asarray(y);Nn=len(y);pr=y.mean();out=[]
    for t in ts:
        pp=p>=t;TP=np.sum(pp&(y==1));FP=np.sum(pp&(y==0));w=t/(1-t)
        out.append({'t':round(t,2),'model':round(TP/Nn-FP/Nn*w,4),'all':round(pr-(1-pr)*w,4)})
    d=pd.DataFrame(out); d['wins']=d['model']>d[['all']].assign(none=0).max(axis=1); return d
def or_table(X,y):
    Xs=X.copy()
    for c in [c for c in CONT if c in Xs.columns]:
        s=pd.to_numeric(Xs[c],errors='coerce'); Xs[c]=(s-s.mean())/s.std()
    m=LogisticRegression(penalty='l2',C=1.0,solver='liblinear',max_iter=5000,random_state=42).fit(Xs.values,y)
    p=m.predict_proba(Xs.values)[:,1]; W=p*(1-p); Xd=np.column_stack([np.ones(len(Xs)),Xs.values])
    try: se=np.sqrt(np.diag(np.linalg.inv(Xd.T@(Xd*W[:,None]))))[1:]
    except Exception: se=np.full(Xs.shape[1],np.nan)
    co=m.coef_[0]
    return pd.DataFrame({'predictor':list(Xs.columns),'OR':np.exp(co),'lo':np.exp(co-1.96*se),'hi':np.exp(co+1.96*se)}).sort_values('OR')
def _cvauc(model,Xd,y,cv):
    return roc_auc_score(y,cross_val_predict(model,Xd,y,cv=cv,method='predict_proba')[:,1])
def panel(X,y,C=1.0,rs=42):
    cv=StratifiedKFold(5,shuffle=True,random_state=rs); Xi=X.fillna(X.median()); res={}
    # always-on, stable models
    try: res['logistic (tuned)']=_cvauc(pipe_fixed(list(X.columns),'l2',C),X,y,cv)
    except Exception as e: print('  logistic failed:',e)
    try: res['lasso']=_cvauc(pipe_fixed(list(X.columns),'l1',0.5),X,y,cv)
    except Exception as e: print('  lasso failed:',e)
    try: res['random forest']=_cvauc(RandomForestClassifier(n_estimators=200,max_depth=4,n_jobs=1,random_state=rs),Xi,y,cv)
    except Exception as e: print('  random forest failed:',e)
    # heavy models only if explicitly enabled
    if HAVE_HGB:
        try: res['grad boosting']=_cvauc(HistGradientBoostingClassifier(max_iter=150,learning_rate=0.05,max_depth=3,random_state=rs),Xi,y,cv)
        except Exception as e: print('  grad boosting failed:',e)
    if HAVE_XGB:
        try: res['XGBoost']=_cvauc(XGBClassifier(n_estimators=200,max_depth=3,learning_rate=0.05,eval_metric='logloss',n_jobs=1,random_state=rs,verbosity=0),Xi,y,cv)
        except Exception as e: print('  XGBoost failed:',e)
    if HAVE_EBM:
        try: res['EBM']=_cvauc(ExplainableBoostingClassifier(random_state=rs),Xi,y,cv)
        except Exception as e: print('  EBM failed:',e)
    return res
print('functions ready')

In [ ]:
rows=[]; store={}
for o in OUTCOMES:
    cc=df[PREOP+OPER_CORE+[o]].dropna(); y=cc[o].astype(int)
    if len(cc)<80 or y.nunique()<2: print('skip',o); continue
    asa=evalm(cc[['asascore']],y); pre=evalm(cc[PREOP],y); per=evalm(cc[PREOP+OPER_CORE],y)
    perC=best_C(cc[PREOP+OPER_CORE],y)
    n=min(len(pre['boot']),len(per['boot'])); d=per['boot'][:n]-pre['boot'][:n]
    pevent=(1-per['cvp']) if o=='textbook_failure' else per['cvp']; yevent=(1-per['y']) if o=='textbook_failure' else per['y']
    # for textbook_failure the model already predicts failure, so use directly:
    dd=dca(per['y'],per['cvp'],np.arange(0.05,0.51,0.05))
    store[o]={'pre':pre,'per':per,'dca':dd,'cc':cc}
    rows.append({'outcome':o,'N':len(cc),'events':int(y.sum()),'ASA':round(asa['auc'],3),
                 'preop':round(pre['auc'],3),'periop':round(per['auc'],3),'gain':round(per['auc']-pre['auc'],3),
                 'gain_ci':f"[{np.percentile(d,2.5):.3f},{np.percentile(d,97.5):.3f}]",'p_periop_better':round(np.mean(d>0),3),
                 'periop_ci':f"[{per['lo']:.2f},{per['hi']:.2f}]",'brier':round(per['brier'],3),
                 'cal_slope':round(per['cal'],2),'mp_vs_obs':f"{per['mp']:.2f}/{per['obs']:.2f}",'dca_wins':int(dd['wins'].sum())}); store[o]['perC']=perC; print('done',o)
summary=pd.DataFrame(rows); summary.to_csv(OUT/'periop_vs_preop_summary.csv',index=False)
print('\n'+summary[['outcome','N','events','ASA','preop','periop','gain','p_periop_better','cal_slope','dca_wins']].to_string(index=False))

In [ ]:
panel_rows=[]
for o in OUTCOMES:
    cc=store[o]['cc']; y=cc[o].astype(int); X=cc[PREOP+OPER_CORE]
    r=panel(X,y,C=store[o]['perC']); r['outcome']=o; panel_rows.append(r); print('panel done',o)
panel_df=pd.DataFrame(panel_rows).set_index('outcome').round(3)
panel_df.to_csv(OUT/'model_family_panel.csv')
print(panel_df.to_string())

In [ ]:
or_tables={}
for o in HEADLINE:
    cc=store[o]['cc']; y=cc[o].astype(int); X=cc[PREOP+OPER_CORE].astype(float)
    t=or_table(X,y); or_tables[o]=t
    print(f"\n=== {o}: standardised odds ratios ==="); print(t.to_string(index=False))

In [ ]:
# 7a discrimination comparison with confidence intervals
labels=[r['outcome'] for r in rows]; x=np.arange(len(labels)); wd=0.25
fig,ax=plt.subplots(figsize=(11,5))
for k,(key,off) in enumerate([('ASA',-wd),('preop',0),('periop',wd)]):
    vals=[r[key] for r in rows]
    ax.bar(x+off,vals,wd,label=key)
ax.axhline(0.5,ls=':',color='grey'); ax.set_xticks(x); ax.set_xticklabels(labels,rotation=20,ha='right')
ax.set_ylabel('cross validated AUC'); ax.set_ylim(0.4,0.75); ax.set_title('Discrimination: ASA vs preoperative vs perioperative'); ax.legend()
plt.tight_layout(); plt.savefig(OUT/'discrimination_comparison.png',dpi=120); plt.close(fig)

# 7b model family panel
fig,ax=plt.subplots(figsize=(12,5)); models=[c for c in panel_df.columns]; xm=np.arange(len(panel_df.index)); w=0.8/len(models)
for i,mname in enumerate(models): ax.bar(xm+i*w,panel_df[mname].values,w,label=mname)
ax.set_xticks(xm+0.4-w/2); ax.set_xticklabels(panel_df.index,rotation=20,ha='right'); ax.axhline(0.5,ls=':',color='grey')
ax.set_ylabel('cross validated AUC'); ax.set_ylim(0.4,0.75); ax.set_title('Model family comparison (perioperative features)'); ax.legend(fontsize=7,ncol=3)
plt.tight_layout(); plt.savefig(OUT/'model_family_comparison.png',dpi=120); plt.close(fig)

# 7c calibration (all outcomes)
fig,axes=plt.subplots(2,3,figsize=(15,8))
for ax,o in zip(axes.ravel(),OUTCOMES):
    per=store[o]['per']; frac,mp=calibration_curve(per['y'],per['cvp'],n_bins=8,strategy='quantile')
    ax.plot([0,1],[0,1],ls=':',color='grey'); ax.plot(mp,frac,marker='o')
    ax.set_title(f"{o}\nAUC {per['auc']:.2f}, slope {per['cal']:.2f}"); ax.set_xlabel('predicted'); ax.set_ylabel('observed'); ax.set_xlim(0,1); ax.set_ylim(0,1)
plt.tight_layout(); plt.savefig(OUT/'calibration_periop.png',dpi=120); plt.close(fig)

# 7d decision curves (textbook is failure, so reads correctly)
fig,axes=plt.subplots(2,3,figsize=(15,8))
for ax,o in zip(axes.ravel(),OUTCOMES):
    d=store[o]['dca']; ax.plot(d['t'],d['model'],marker='o',label='periop model'); ax.plot(d['t'],d['all'],ls='--',label='treat all'); ax.axhline(0,ls=':',color='grey',label='treat none')
    ax.set_title(o); ax.set_xlabel('threshold'); ax.set_ylabel('net benefit'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT/'decision_curves.png',dpi=120); plt.close(fig)

# 7e forest plots for the headline models
for o in HEADLINE:
    t=or_tables[o]; fig,ax=plt.subplots(figsize=(7,6))
    ax.errorbar(t['OR'],range(len(t)),xerr=[t['OR']-t['lo'],t['hi']-t['OR']],fmt='o',capsize=3)
    ax.axvline(1,ls=':',color='grey'); ax.set_yticks(range(len(t))); ax.set_yticklabels(t['predictor']); ax.set_xlabel('odds ratio (95% CI)')
    ax.set_title(f'{o}: adjusted odds ratios'); plt.tight_layout(); plt.savefig(OUT/f'forest_{o}.png',dpi=120); plt.close(fig)
print('saved figures:', sorted(p.name for p in OUT.glob('*.png')))

In [ ]:
# 8a predictor collinearity, VIF and univariable AUC (Edwin, 6 June): resolve overlapping predictors
from sklearn.linear_model import LinearRegression
def vif_table(Xnum):
    cols=list(Xnum.columns); out=[]
    for c in cols:
        yv=Xnum[c].values; Xo=Xnum.drop(columns=[c]).values
        r2=LinearRegression().fit(Xo,yv).score(Xo,yv); out.append((c,round(1/max(1e-6,1-r2),2)))
    return pd.DataFrame(out,columns=['predictor','VIF']).sort_values('VIF',ascending=False)
def uni_auc(col,outcome):
    cc=df[[col,outcome]].dropna(); y=cc[outcome].astype(int)
    if y.nunique()<2 or len(cc)<50: return np.nan
    x=pd.to_numeric(cc[col],errors='coerce').values
    if np.unique(x[~np.isnan(x)]).size<2: return np.nan
    a=roc_auc_score(y,x); return round(max(a,1-a),3)   # direction-free univariable discrimination
preop_num=df[PREOP].apply(pd.to_numeric,errors='coerce')
cm=preop_num.corr(method='spearman').round(2)
print('Spearman correlation among preoperative predictors:'); print(cm.to_string())
pairs=[(a,b,float(cm.loc[a,b])) for i,a in enumerate(cm.columns) for b in cm.columns[i+1:] if abs(cm.loc[a,b])>0.7]
print('\npairs with |correlation| > 0.7:', pairs if pairs else 'none')
print('\nVIF (variance inflation, >5 suggests collinearity):'); print(vif_table(preop_num.fillna(preop_num.median())).to_string(index=False))
ua=pd.DataFrame({o:[uni_auc(c,o) for c in PREOP] for o in HEADLINE},index=PREOP); ua['mean']=ua.mean(axis=1).round(3)
print('\nunivariable AUC by predictor (headline outcomes):'); print(ua.sort_values('mean',ascending=False).to_string())
# clinically overlapping groups to resolve (6 June): fitness/comorbidity and neoadjuvant
GROUPS={'fitness':['asascore','charlson_cci','comlong'],'neoadjuvant':['neoadj_received','neoadj_chemoradiation']}
drop=set()
for g,members in GROUPS.items():
    members=[m for m in members if m in PREOP]
    if not members: continue
    best=ua.loc[members,'mean'].idxmax()
    print(f"\ngroup {g}: best univariable = {best} | members {ua.loc[members,'mean'].to_dict()}")
    for m in members:                       # drop only if highly correlated (>0.7) with the retained best
        if m!=best and abs(cm.loc[m,best])>0.7: drop.add(m)
PREOP_DEDUP=[c for c in PREOP if c not in drop]
print('\ndropped for collinearity (>0.7 within an overlap group):', sorted(drop) if drop else 'none')
print('PREOP_DEDUP', len(PREOP_DEDUP), PREOP_DEDUP)

In [ ]:
# 8b does removing the overlapping predictors change discrimination (preop full vs de-duplicated)
rows2=[]
for o in OUTCOMES:
    cc=df[PREOP+OPER_CORE+[o]].dropna(); y=cc[o].astype(int)
    if len(cc)<80 or y.nunique()<2: continue
    full=evalm(cc[PREOP],y,tuned=False); ded=evalm(cc[PREOP_DEDUP],y,tuned=False)
    rows2.append({'outcome':o,'preop_full':round(full['auc'],3),'preop_dedup':round(ded['auc'],3),
                  'delta':round(ded['auc']-full['auc'],3),'k_full':len(PREOP),'k_dedup':len(PREOP_DEDUP)})
dedup_df=pd.DataFrame(rows2); dedup_df.to_csv(OUT/'preop_dedup_comparison.csv',index=False)
print(dedup_df.to_string(index=False))

In [ ]:
# 8c baseline variables vs others, nested models (baseline = routine core; confirm set with Edwin)
BASELINE=[c for c in ['age_at_surgery','sex_male','asascore'] if c in df.columns]
ADD_COMORBID=[c for c in ['charlson_cci','comlong'] if c in df.columns]
ADD_TUMOUR=[c for c in ['neoadj_received','neoadj_chemoradiation','ct_stage','cn_stage','histology_adeno'] if c in df.columns]
ADD_OPER=[c for c in OPER_CORE]
STEPS=[('baseline',BASELINE),('+comorbidity',BASELINE+ADD_COMORBID),
       ('+tumour_treatment',BASELINE+ADD_COMORBID+ADD_TUMOUR),
       ('+operative',BASELINE+ADD_COMORBID+ADD_TUMOUR+ADD_OPER)]
allcols=BASELINE+ADD_COMORBID+ADD_TUMOUR+ADD_OPER
nested_rows=[]
for o in OUTCOMES:
    cc=df[allcols+[o]].dropna(); y=cc[o].astype(int)
    if len(cc)<80 or y.nunique()<2: continue
    row={'outcome':o,'N':len(cc),'events':int(y.sum())}
    for name,cols in STEPS: row[name]=round(evalm(cc[cols],y,tuned=False)['auc'],3)
    nested_rows.append(row)
nested_df=pd.DataFrame(nested_rows); nested_df.to_csv(OUT/'baseline_vs_others.csv',index=False)
print('baseline = age, sex, ASA (routine core)'); print(nested_df.to_string(index=False))

In [ ]:
# 8d prolonged stay handled by surgery year (protocol changes yearly, 6 June)
# per-year separate models are underpowered, so adjust the pooled model for year and report within-year discrimination
o='los_prolonged'
cc=df[PREOP+OPER_CORE+['surgery_year',o]].dropna(); y=cc[o].astype(int)
base=evalm(cc[PREOP+OPER_CORE],y,tuned=False)
Xadj=cc[PREOP+OPER_CORE].copy(); Xadj['surgery_year']=cc['surgery_year'].astype(float)
adj=evalm(Xadj,y,tuned=False)
print('prolonged stay: perioperative AUC %.3f | + surgery_year adjustment AUC %.3f'%(base['auc'],adj['auc']))
cvp=base['cvp']; yr=cc['surgery_year'].values; yv=y.values
peryr=[]
for u in sorted(set(yr)):
    msk=yr==u
    if msk.sum()>=20 and len(np.unique(yv[msk]))>1:
        peryr.append({'year':int(u),'n':int(msk.sum()),'events':int(yv[msk].sum()),
                      'rate':round(float(yv[msk].mean()),3),'auc_within_year':round(roc_auc_score(yv[msk],cvp[msk]),3)})
peryr_df=pd.DataFrame(peryr); peryr_df.to_csv(OUT/'prolonged_by_year.csv',index=False)
print('\nper-year discrimination of the pooled perioperative model (within-year AUC):'); print(peryr_df.to_string(index=False))